<a href="https://colab.research.google.com/github/kousheel05/TASK_AIML-/blob/main/feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================
# FEATURE ENGINEERING ASSIGNMENT
# ============================================

# Import Libraries
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder

# ============================================
# Load Dataset
# ============================================
from google.colab import files
uploaded = files.upload()
df = pd.read_csv("employee_performance.csv")

print("Original Dataset")
print(df.head())

# ============================================
# Create Required Columns
# (Because the dataset doesn't contain them)
# ============================================

np.random.seed(42)

# City
cities = ["Hyderabad", "Bengaluru", "Chennai", "Mumbai", "Pune"]
df["City"] = np.random.choice(cities, len(df))

# Purchase Amount
df["Purchase_Amount"] = np.random.randint(1000, 50000, len(df))

# Date
df["Date"] = pd.date_range(start="2024-01-01", periods=len(df))

# Rating
df["Rating"] = df["Performance_Score"]

# ============================================
# Introduce Missing Values (For Practice)
# ============================================

df.loc[5, "Salary"] = np.nan
df.loc[10, "Age"] = np.nan
df.loc[15, "Gender"] = np.nan
df.loc[20, "City"] = np.nan

print("\nDataset After Adding Required Columns")
print(df.head())

# ============================================
# Problem 1 : Handling Missing Values
# ============================================

print("\nMissing Values Before")
print(df.isnull().sum())

numerical = df.select_dtypes(include=np.number).columns
categorical = df.select_dtypes(include="object").columns

for col in numerical:
    df[col].fillna(df[col].mean(), inplace=True)

for col in categorical:
    df[col].fillna(df[col].mode()[0], inplace=True)

print("\nMissing Values After")
print(df.isnull().sum())

# ============================================
# Problem 2 : Feature Scaling
# ============================================

print("\nBefore Scaling")
print(df[["Age", "Salary"]].head())

standard = StandardScaler()
df["Salary_Standardized"] = standard.fit_transform(df[["Salary"]])

minmax = MinMaxScaler()
df["Age_MinMax"] = minmax.fit_transform(df[["Age"]])

print("\nAfter Scaling")
print(df[["Age", "Age_MinMax", "Salary", "Salary_Standardized"]].head())

# ============================================
# Problem 3 : Encoding
# ============================================

label = LabelEncoder()

df["Gender_Label"] = label.fit_transform(df["Gender"])

city_encoded = pd.get_dummies(df["City"], prefix="City")

df = pd.concat([df, city_encoded], axis=1)

print("\nDataset After Encoding")
print(df.head())

# ============================================
# Problem 4 : Feature Creation
# ============================================

df["Total_Spending"] = df["Salary"] + df["Purchase_Amount"]

print("\nFeature Creation")
print(df[["Salary", "Purchase_Amount", "Total_Spending"]].head())

# ============================================
# Problem 5 : Date Feature Extraction
# ============================================

df["Date"] = pd.to_datetime(df["Date"])

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["Weekday"] = df["Date"].dt.day_name()

print("\nDate Features")
print(df[["Date", "Year", "Month", "Day", "Weekday"]].head())

# ============================================
# Problem 6 : Binning
# ============================================

bins = [0, 25, 45, 100]

labels = ["Young", "Adult", "Senior"]

df["Age_Group"] = pd.cut(df["Age"], bins=bins, labels=labels)

print("\nAge Group")
print(df[["Age", "Age_Group"]].head(10))

# ============================================
# Problem 7 : Outlier Detection using IQR
# ============================================

Q1 = df["Salary"].quantile(0.25)

Q3 = df["Salary"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR

upper_limit = Q3 + 1.5 * IQR

salary_outliers = df[
    (df["Salary"] < lower_limit) |
    (df["Salary"] > upper_limit)
]

print("\nSalary Outliers")
print(salary_outliers)

# ============================================
# Problem 8 : Feature Interaction
# ============================================

df["Experience_Salary"] = df["Experience_Years"] * df["Salary"]

print("\nExperience Salary")
print(df[["Experience_Years", "Salary", "Experience_Salary"]].head())

print("""
Importance:
Experience_Salary combines employee experience and salary,
which helps machine learning models learn hidden relationships
between salary growth and work experience.
""")

# ============================================
# Problem 9 : Feature Selection
# ============================================

temp = df.copy()

le = LabelEncoder()

temp["Department"] = le.fit_transform(temp["Department"])
temp["City"] = le.fit_transform(temp["City"])
temp["Weekday"] = le.fit_transform(temp["Weekday"])
temp["Age_Group"] = le.fit_transform(temp["Age_Group"].astype(str))
temp["Employee_ID"] = le.fit_transform(temp["Employee_ID"])

correlation = temp.corr(numeric_only=True)

print("\nCorrelation with Purchase_Amount")
print(correlation["Purchase_Amount"].sort_values(ascending=False))

least_feature = (
    correlation["Purchase_Amount"]
    .drop("Purchase_Amount")
    .abs()
    .idxmin()
)

print("\nLeast Important Feature:", least_feature)

final_df = df.drop(columns=[least_feature])

print("\nFinal Dataset")
print(final_df.head())

# ============================================
# Assignment Completed
# ============================================

print("\nFeature Engineering Assignment Completed Successfully!")

Saving employee_performance.csv to employee_performance.csv
Original Dataset
  Employee_ID  Age  Gender  Department  Experience_Years  Performance_Score  \
0       E0524   49  Female  Operations               5.6               67.8   
1       E0603   55  Female       Sales               5.5                NaN   
2       E0527   29  Female       Sales              10.7               68.8   
3       E0032   30  Female  Operations               9.8                NaN   
4       E0617   58    Male       Sales              11.5               72.8   

    Salary  
0  33013.0  
1  42640.0  
2  72180.0  
3  65750.0  
4  74830.0  

Dataset After Adding Required Columns
  Employee_ID   Age  Gender  Department  Experience_Years  Performance_Score  \
0       E0524  49.0  Female  Operations               5.6               67.8   
1       E0603  55.0  Female       Sales               5.5                NaN   
2       E0527  29.0  Female       Sales              10.7               68.8   
3       E00

/tmp/ipykernel_637/2637962949.py:66: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mean(), inplace=True)
/tmp/ipykernel_637/2637962949.py:69: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try usin